# 01 — Load and Explore

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from scipy import stats
from sklearn.linear_model import LogisticRegression
from pathlib import Path

In [2]:
files = sorted(Path("data/raw").glob("*.csv"))

print(files)

[PosixPath('data/raw/2014-15.csv'), PosixPath('data/raw/2015-16.csv'), PosixPath('data/raw/2016-17.csv'), PosixPath('data/raw/2017-18.csv'), PosixPath('data/raw/2018-19.csv'), PosixPath('data/raw/2019-20.csv'), PosixPath('data/raw/2020-21.csv'), PosixPath('data/raw/2021-22.csv'), PosixPath('data/raw/2022-23.csv'), PosixPath('data/raw/2023-24.csv'), PosixPath('data/raw/2024-25.csv'), PosixPath('data/raw/2025-26.csv')]


In [3]:
# Columns we need for analysis
dfs = [pd.read_csv(f, usecols=["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]).assign(Season=f.stem) for f in files]
# Ignore index true to prevent repeated indices
combined = pd.concat(dfs, ignore_index=True)
# Consistent date format
combined["Date"] = pd.to_datetime(combined["Date"], dayfirst=True)

# Inspect
display(combined.shape)
display(combined.head())

/var/folders/rk/8158ycq96b9fbxc855vg66jc0000gn/T/ipykernel_58178/1626321071.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  combined["Date"] = pd.to_datetime(combined["Date"], dayfirst=True)


(4561, 7)

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,Season
0,2014-08-16,Arsenal,Crystal Palace,2.0,1.0,H,2014-15
1,2014-08-16,Leicester,Everton,2.0,2.0,D,2014-15
2,2014-08-16,Man United,Swansea,1.0,2.0,A,2014-15
3,2014-08-16,QPR,Hull,0.0,1.0,A,2014-15
4,2014-08-16,Stoke,Aston Villa,0.0,1.0,A,2014-15


In [4]:
# Investigate potential date parsing issue
print(combined['Season'].value_counts().sort_index())
print(combined['Date'].isna().sum())
print(combined.isna().sum())

Season
2014-15    381
2015-16    380
2016-17    380
2017-18    380
2018-19    380
2019-20    380
2020-21    380
2021-22    380
2022-23    380
2023-24    380
2024-25    380
2025-26    380
Name: count, dtype: int64
1
Date        1
HomeTeam    1
AwayTeam    1
FTHG        1
FTAG        1
FTR         1
Season      0
dtype: int64


2014-15 has one extra row which was empty (trailing blank line in source CSV). Dropped it, hence the final dataset has 4560 matches across 12 seasons as expected. 

In [5]:
# Inspect bad rows (1 missing value under every col but Season)
combined[combined['Date'].isna()]
# Remove row where data is missing
combined = combined.dropna(subset=['Date'])

# Inspect
print(combined.shape)
print(combined.isna().sum())

(4560, 7)
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
Season      0
dtype: int64


In [6]:
# Inspect and verify
print(combined['Date'].min(), combined['Date'].max())
print(combined.dtypes)
print(sorted(combined['HomeTeam'].unique()))
print(sorted(combined['AwayTeam'].unique()))

# Save
combined.to_parquet("data/processed/matches.parquet")
print("Saved.")

2014-08-16 00:00:00 2026-05-24 00:00:00
Date        datetime64[us]
HomeTeam               str
AwayTeam               str
FTHG               float64
FTAG               float64
FTR                    str
Season                 str
dtype: object
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'QPR', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'QPR', 'Sheffield 

In [7]:
# Verify file exists
test = pd.read_parquet("data/processed/matches.parquet")
print(test.shape)
print(test.dtypes)

(4560, 7)
Date        datetime64[us]
HomeTeam               str
AwayTeam               str
FTHG               float64
FTAG               float64
FTR                    str
Season                 str
dtype: object
